<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Compliance phrases and keyword spotting

**[Compliance phrases and keyword spotting](https://learning.nextia-ai.com/courses/speech/m06/compliance-phrases-and-keyword-spotting/)** · Speech AI: Transcribe, Understand and Speak · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** check that the agent said the recording notice: with an exact search, with a fuzzy match that accepts other wordings and recognition errors, and on recognised transcripts. You will measure recall and false alarms, and use the train and validation labels where the test calls have too few calls without the notice.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no model download. |
| **You should know** | The call records, from [Summaries and follow-ups](https://learning.nextia-ai.com/courses/speech/m06/summaries-and-follow-ups/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m06/compliance-phrases-and-keyword-spotting/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M06-L03-compliance-phrases-and-keyword-spotting/compliance-phrases-and-keyword-spotting-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/recordings/asr/whisper-small.jsonl": "67f27c0e6f5aa73d0e67cdb4b69c0e2d22c5f03a5dd74be14cbd8db0df17cb3f",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recording notice in the labels

Larkfield's agents must say the recording notice at the start of each call: "This call is recorded for quality and training." Some say a fair other wording (the label `variant`), and some forget it (`missing`). Every call is synthetic: open text-to-speech voices read scripts that we wrote.

In [ ]:
calls = load_calls()
public = list(calls.values())                        # train 150, validation 50, test 60
test = [c for c in public if c["split"] == "test"]
agent_text = {c["call_id"]: reference(c, "agent") for c in public}   # what the agent said, exactly
recognised = {}
for line in open("data/recordings/asr/whisper-small.jsonl", encoding="utf-8"):
    row = json.loads(line)
    if row["kind"] == "phone":
        recognised[row["call_id"]] = row["text"]   # both speakers, no labels
from collections import Counter
print(Counter(c["notice"] for c in test), " (test)")
print(Counter(c["notice"] for c in public), " (all public calls)")

> **Check.** Only 3 of the 60 test calls miss the notice. Across all 260 public calls, 39 miss it: the train calls have transcripts and labels, so you can use them for this check.

## 2. The exact search

> **Predict.** Call C-0064's agent says "calls may be recorded for training and quality purposes". Does the exact search find the notice?

In [ ]:
NOTICE = {"en": "this call is recorded for quality and training",
          "fr": "cet appel est enregistré pour la qualité et la formation"}

def notice_exact(text, language):
    """True when the notice appears letter for letter, after the basic normalisation."""
    return NOTICE[language] in cp.normalise(text, "basic", language)

def score(find, calls_list, texts):
    """(notices found, calls that said it, false alarms) for a search function."""
    said = [c["notice"] != "missing" for c in calls_list]
    found = [find(texts[c["call_id"]], c["language"]) for c in calls_list]
    return (sum(f and s for f, s in zip(found, said)), sum(said),
            sum(f and not s for f, s in zip(found, said)))

print(agent_text["C-0064"][:150])
print("exact:", notice_exact(agent_text["C-0064"], "en"))
print("test calls, exact transcripts (found, said, false alarms):", score(notice_exact, test, agent_text))

> **Check.** No: the words are there, in another order. The exact search finds (48, 57, 0) (found, said, false alarms) on the test calls: it misses every fair variant.

## 3. Similarity, by hand

Whisper small heard the French notice of call C-0020 as "ce tapel" instead of "cet appel". The similarity of two texts counts the letters that appear in both, in the same order: two times that number, divided by the two lengths added.

In [ ]:
import difflib
heard = "ce tapel est enregistré pour la qualité et la formation"   # Whisper small, call C-0020
target = NOTICE["fr"]
m = difflib.SequenceMatcher(None, heard, target)
same = sum(block.size for block in m.get_matching_blocks())
print(f"letters in the same order: {same}; lengths {len(heard)} and {len(target)}")
print(f"similarity = 2 x {same} / ({len(heard)} + {len(target)}) = {m.ratio():.3f}")
print("exact:", target in heard)

> **Check.** 54 letters in the same order, similarity 0.973: almost the same text, but the exact search says no.

## 4. Fuzzy match

The project's fuzzy match accepts the notice in two ways: all its key words, in any order; or a window of words that is similar to the notice (0.6 or more) and has at least one key word.

In [ ]:
NOTICE_WORDS = {"en": {"recorded", "quality", "training"},
                "fr": {"enregistré", "enregistrés", "qualité", "formation"}}

def best_window(text, language):
    """The highest similarity between the notice and any window of words as long as the notice."""
    words = cp.normalise(text, "basic", language).split()
    n = len(NOTICE[language].split())
    return max((difflib.SequenceMatcher(None, " ".join(words[i:i + n]), NOTICE[language]).ratio()
                for i in range(max(1, len(words) - n + 1))), default=0)

def notice_fuzzy(text, language, threshold=0.6):
    """The key words in any order, or a window similar to the notice (with at least one key word)."""
    words = set(cp.normalise(text, "basic", language).split())
    if language == "en" and {"recorded", "quality", "training"} <= words:
        return True
    if language == "fr" and {"qualité", "formation"} <= words and words & {"enregistré", "enregistrés"}:
        return True
    return best_window(text, language) >= threshold and bool(words & NOTICE_WORDS[language])

print("C-0064:", notice_fuzzy(agent_text["C-0064"], "en"), " best window", round(best_window(agent_text["C-0064"], "en"), 3))
print("test calls, exact transcripts:", score(notice_fuzzy, test, agent_text))

> **Check.** C-0064 passes by its key words. Its best window is only 0.629, just over the threshold. On the test calls: (57, 57, 0).

## 5. On Whisper's transcripts

In production, the search reads Whisper's text, with both speakers and the recognition errors.

In [ ]:
print(recognised["C-0020"][:120])
print("Whisper small, exact:", score(notice_exact, test, recognised))
print("Whisper small, fuzzy:", score(notice_fuzzy, test, recognised))

> **Check.** Exact (41, 57, 0), fuzzy (57, 57, 0). Whisper writes "Lockfield" and "ce tapel"; the fuzzy match does not mind.

## 6. False alarms need calls without the notice

A **false alarm** here is a call without the notice that the search marks as compliant. With 3 such test calls, you cannot measure it well. Use all 260 public calls (39 without the notice).

In [ ]:
print("all public calls, exact:", score(notice_exact, public, agent_text))
print("all public calls, fuzzy:", score(notice_fuzzy, public, agent_text))

> **Check.** Exact (183, 221, 0), fuzzy (221, 221, 0): no false alarm in 39 calls. But in these scripts, an agent who forgets the notice says nothing like it. Real agents say things that are close.

## 7. Failure case: close is not the same

Two sentences that we made up on purpose:

In [ ]:
for sentence in ["This call is not recorded for quality and training.",
                 "Was my last call recorded? The training of your staff is poor quality."]:
    print(notice_exact(sentence, "en"), notice_fuzzy(sentence, "en"), "<-", sentence)

> **Check.** The fuzzy match says yes to both: the key words are all there. The first one says the opposite of the notice. A fuzzy match must be tested on such near misses, and a person should listen to a sample of the calls it passes.

## 8. Your turn: the window threshold

> **Your turn.** Without the key-word rule, how many notices does the window alone find on Whisper small's test transcripts? For each threshold 0.5, 0.6, 0.7, 0.8 and 0.9, compute the share of the calls that said the notice (`notice != "missing"`) whose `best_window` is at least the threshold. Put the results in a dictionary `window_recall` (threshold: share, rounded to 3 decimals).

In [ ]:
window_recall = {}
said = [c for c in test if c["notice"] != "missing"]
for t in (0.5, 0.6, 0.7, 0.8, 0.9):
    window_recall[t] = round(sum(best_window(recognised[c["call_id"]], c["language"]) >= t for c in said) / len(said), 3)
print(window_recall)

In [ ]:
expect_hash('window_recall', window_recall, '2f555423017cb3ba')

> **Check.** At 0.5 the window alone finds every notice; at 0.6 it misses one; from 0.7 on, it finds 48 of 57, as many as there are exact notices. The nine variants are never close enough, letter by letter. That is why the key-word rule is there. A higher threshold would also give fewer false alarms, but this set has too few near misses to show it.

## Next

You checked the recording notice with an exact search and a fuzzy match, on exact and recognised transcripts, and measured recall and false alarms. The [lesson page](https://learning.nextia-ai.com/courses/speech/m06/compliance-phrases-and-keyword-spotting/) also shows keyword spotting in the audio, and has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [What an IVR is, and why companies still use one](https://learning.nextia-ai.com/courses/speech/m07/what-an-ivr-is/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m06/compliance-phrases-and-keyword-spotting/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.